In [ ]:
import argparse
import torch
import numpy as np
import random

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import os
import sys
import json
from datasets import load_dataset
from predict_module import sft_dataloader

from utils.prompts import PREDICT_INSTRUCTION
from utils.fewshots import PREDICT_EXAMPLES


# Cấu hình tham số cho huấn luyện
args = argparse.Namespace(
    
    wandb=False,  # Tắt logging với Weights & Biases
    data_path="./data/DeepSeekLLM_top1_stock_merge_sample.json",  # Đường dẫn file dữ liệu
    output_path="./saved_models/lora-DeepSeek-R1-Distill-Qwen",  # Thư mục lưu mô hình LoRA
    model_path="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",  # Mô hình DeepSeek
    eval_steps=200,  # Số bước đánh giá
    save_steps=200,  # Số bước lưu checkpoint
    resume_from_supervised_checkpoint=None,  # Không resume từ checkpoint
    ignore_data_skip="False",  # Không bỏ qua dữ liệu khi resume
    num_reflect_trials=2,  # Số lần thử phản ánh
    datasets_dir="./datasets/",  # Thư mục datasets
    local_rank=0,  # Rank cục bộ cho DDP
    resume_from_reward_checkpoint=False,  # Không resume từ reward checkpoint
    deepspeed=None,  # Không dùng DeepSpeed
    per_device_train_batch_size=4,  # Batch size huấn luyện trên mỗi GPU
    per_device_eval_batch_size=4,  # Batch size đánh giá trên mỗi GPU
    reward_gradient_accumulation_steps=8,  # Số bước tích lũy gradient cho reward
    reward_learning_rate=3e-5,  # Learning rate cho reward
    weight_decay=0.001,  # Trọng số giảm dần
    reward_base_model="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",  # Mô hình reward
    bf16=False,  # Sử dụng fp16 thay vì bf16
    num_train_epochs=2,  # Số epoch huấn luyện
    train_subset=100000,  # Số mẫu huấn luyện
    eval_subset=50000,  # Số mẫu đánh giá
    gradient_checkpointing=True,  # Bật gradient checkpointing để tiết kiệm VRAM
    optim="adamw_torch",  # Optimizer AdamW từ PyTorch
    lr_scheduler_type="cosine",  # Lịch trình learning rate kiểu cosine
    reward_adapter="./saved_models/reward_model_deepseek-r1-distill-qwen",  # Adapter reward
    rl_base_model="./saved_models/lora-DeepSeek-R1-Distill-Qwen-adapter-merged",  # Mô hình RL
    tokenizer_name="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",  # Tokenizer
    reward_model_name="./saved_models/reward_model_deepseek-r1-distill-qwen-adapter-merged",  # Mô hình reward merged
    log_with=None,  # Không dùng logging cụ thể
    rl_learning_rate=2e-5,  # Learning rate cho RL
    output_max_length=256,  # Độ dài đầu ra tối đa
    mini_batch_size=4,  # Kích thước mini-batch
    batch_size=128,  # Kích thước batch tổng
    ppo_epochs=4,  # Số epoch cho PPO
    rl_gradient_accumulation_steps=32,  # Số bước tích lũy gradient cho RL
    adafactor=False,  # Không dùng Adafactor
    early_stopping=True,  # Bật early stopping
    target_kl=0.1,  # KL target cho RL
    reward_baseline=0,  # Baseline cho reward
    batched_gen=True,  # Tạo batch
    save_freq=100,  # Tần suất lưu
    output_dir="./saved_models/tuning_deepseek_r1_distill_qwen_checkpoints/",  # Thư mục lưu checkpoint
    seed=0,  # Seed cho RL
    num_shots=4,  # Số shots cho few-shot
    save_dir="results/"  # Thư mục lưu kết quả
)

os.makedirs(args.datasets_dir, exist_ok=True)

print("Args in experiment:")
print(args)

args.data_path


Args in experiment:
Namespace(wandb=False, data_path='./data/DeepSeekLLM_top1_stock_merge_sample.json', output_path='./saved_models/lora-DeepSeek-R1-Distill-Qwen', model_path='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', eval_steps=200, save_steps=200, resume_from_supervised_checkpoint=None, ignore_data_skip='False', num_reflect_trials=2, datasets_dir='./datasets/', local_rank=0, resume_from_reward_checkpoint=False, deepspeed=None, per_device_train_batch_size=4, per_device_eval_batch_size=4, reward_gradient_accumulation_steps=8, reward_learning_rate=3e-05, weight_decay=0.001, reward_base_model='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', bf16=False, num_train_epochs=2, train_subset=100000, eval_subset=50000, gradient_checkpointing=True, optim='adamw_torch', lr_scheduler_type='cosine', reward_adapter='./saved_models/reward_model_deepseek-r1-distill-qwen', rl_base_model='./saved_models/lora-DeepSeek-R1-Distill-Qwen-adapter-merged', tokenizer_name='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B

'./data/DeepSeekLLM_top1_stock_merge_sample.json'

## Explain 

In [ ]:
args
import pandas as pd
# Đọc tệp CSV

path_OpenAILLM_top1_stock_train = "../Data/summarized/OpenAILLM_top1_stock_data_train_sample.csv"
# Đọc file CSV vào DataFrame
df_loaded = pd.read_csv(path_OpenAILLM_top1_stock_train)

# Hiển thị nội dung DataFrame
print("Nội dung tệp CSV:")

df_loaded

from explain_module.util import summarize_trial, remove_reflections, save_results#, save_agents
from explain_module.agents import PredictReflectAgent
from utils.llm import OpenAILLM, DeepSeekLLM #, FastChatLLM
import os, json
agent_cls = PredictReflectAgent

MAIN_LLM = DeepSeekLLM()

agents = [agent_cls(row['ticker'], row['summary'], row['target'], predict_llm = MAIN_LLM, reflect_llm= MAIN_LLM) for _, row in df_loaded.iterrows()]
print("Loaded Train Agents.")
agents
i = 1
for agent in agents:
    agent.run()

    if agent.is_correct():
        prompt = agent._build_agent_prompt()
        response = agent.scratchpad.split('Price Movement: ')[-1]
        sample = {"instruction": prompt, "input": "", "output": response}
        with open(args.data_path, 'a') as f:
            f.write(json.dumps(sample) + "\n")
    print(f"Đã xử lý agent thứ {i}/{len(agents)}")
    i=i+1
    
correct, incorrect = summarize_trial(agents)
print(f'Finished Trial 0, Correct: {len(correct)}, Incorrect: {len(incorrect)}')



## Self-Reflection

In [ ]:
# # Train supervised policy
# supervised_finetune(self.args)
# merge_peft_adapter(model_name=self.args.output_path, output_name=self.args.rl_base_model)
print('===================================================')
print('Collect comparison data')

# Collect comparison data
comparison_data = []
for trial in range(args.num_reflect_trials):
    for idx, agent in enumerate([a for a in agents if not a.is_correct()]):
        prev_response = agent.scratchpad.split('Price Movement: ')[-1]
        agent.run()
        if agent.is_correct():
            print(agent._build_agent_prompt(), "\n\n\n")
            prompt = remove_reflections(agent._build_agent_prompt())
            response = agent.scratchpad.split('Price Movement: ')[-1]
            sample = {"user_input": prompt, "completion_a": prev_response, "completion_b": response}
            comparison_data.append(sample)
    correct, incorrect = summarize_trial(agents)
    print(f'Finished Trial {trial+1}, Correct: {len(correct)}, Incorrect: {len(incorrect)}')
os.makedirs(args.datasets_dir, exist_ok=True)
comparison_data_path = os.path.join(args.datasets_dir, "DeepSeekLLM_top1_stock_comparison_data.json")
if comparison_data:
    with open(comparison_data_path, 'w') as f:
        f.write(json.dumps(comparison_data))